# ?? Multilingual Radiology Report NLP Extractor & Weak Supervision

This notebook implements and evaluates a **multilingual clinical NLP extractor** for knee MRI radiology reports.

### Key Objectives:
1. Parse unstructured, multi-lingual free-text reports (English, Spanish, French, Dutch).
2. Apply negation detection and contextual section prioritization (Impression vs. Findings).
3. Validate diagnostic precision and recall against the **58 Gold-Standard** manually annotated studies.
4. Scale our labeled training cohort from **58 to 4,407 studies** via high-confidence pseudo-labeling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

import sys
sys.path.append('..')
from src.report_extractor import (
    extract_study_probabilities, 
    normalize_text, 
    split_clauses, 
    TARGET_COLS
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

DATA_DIR = Path('../data')
print("Extractor module imported successfully.")

## 1. Interactive Parsing & Multi-lingual Demonstration

In [ ]:
sample_english = '''
FINDINGS: Complete tear of the anterior cruciate ligament with adjacent bone contusion at lateral tibial plateau. 
Medial meniscus has complex tear at posterior horn. Lateral meniscus intact. No joint effusion or fracture seen.
IMPRESSION: 
1. Complete ACL tear with bone contusion.
2. Complex medial meniscus tear.
'''

sample_spanish = '''
RESULTADOS: Rotura de menisco interno. Artrosis femorotibial medial moderada. Derrame articular abundante. 
Ligamento cruzado anterior conservado sin signos de rotura. Sin fracturas.
IMPRESION: Meniscopatia interna y gonartrosis medial. Derrame.
'''

sample_french = '''
CONCLUSION : Chondropathie retropatellaire moderee a severe. Epanchement articulaire abondant. 
Pas de dechirure meniscale ni de lesion des ligaments croises.
'''

print("--- English Report Predictions ---")
print(extract_study_probabilities(sample_english))

print("\n--- Spanish Report Predictions ---")
print(extract_study_probabilities(sample_spanish))

print("\n--- French Report Predictions ---")
print(extract_study_probabilities(sample_french))

## 2. Quantitative Benchmark on Gold-Standard Annotations (N=58)

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
labeled_df = train_df[train_df['ACL'].notna()].copy()

gold_preds = [extract_study_probabilities(r) for r in labeled_df['Report']]
gold_pred_df = pd.DataFrame(gold_preds)

metrics_list = []
for target in TARGET_COLS:
    y_true = labeled_df[target].values.astype(int)
    y_prob = gold_pred_df[target].values
    y_bin = (y_prob >= 0.5).astype(int)
    
    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 1.0
    report = classification_report(y_true, y_bin, output_dict=True, zero_division=0)
    
    metrics_list.append({
        'Pathology': target,
        'Gold Positives': int(np.sum(y_true)),
        'Pred Positives': int(np.sum(y_bin)),
        'Precision': round(report['1']['precision'], 3),
        'Recall': round(report['1']['recall'], 3),
        'F1-Score': round(report['1']['f1-score'], 3),
        'AUC-ROC': round(auc, 3)
    })

eval_df = pd.DataFrame(metrics_list)
display(eval_df)

print(f"\n?? Macro-Averaged AUC-ROC: {eval_df['AUC-ROC'].mean():.4f}")
print(f"?? Macro-Averaged F1-Score: {eval_df['F1-Score'].mean():.4f}")

## 3. Evaluation Metric Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(x=eval_df['AUC-ROC'], y=eval_df['Pathology'], palette='viridis', ax=axes[0])
axes[0].set_title("AUC-ROC per Target Abnormality", fontsize=13, fontweight='bold')
axes[0].set_xlim(0.4, 1.0)
axes[0].axvline(eval_df['AUC-ROC'].mean(), color='red', linestyle='--', label=f"Macro-AUC: {eval_df['AUC-ROC'].mean():.3f}")
axes[0].legend()

sns.barplot(x=eval_df['F1-Score'], y=eval_df['Pathology'], palette='mako', ax=axes[1])
axes[1].set_title("F1-Score per Target Abnormality", fontsize=13, fontweight='bold')
axes[1].set_xlim(0.0, 1.0)
axes[1].axvline(eval_df['F1-Score'].mean(), color='orange', linestyle='--', label=f"Macro-F1: {eval_df['F1-Score'].mean():.3f}")
axes[1].legend()

plt.show()

## 4. Full Cohort (N=4,407) Pseudo-Labeled Dataset Overview

In [ ]:
pseudo_df = pd.read_csv(DATA_DIR / 'train_pseudo_labeled.csv')

print(f"Total Studies in Pseudo-Labeled Dataset: {len(pseudo_df):,}")
print("Null values count across all columns:")
print(pseudo_df[[col for col in TARGET_COLS]].isna().sum())

# Compare positive rates between Gold standard and full pseudo-labeled cohort
gold_prev = labeled_df[TARGET_COLS].mean() * 100
full_prev = (pseudo_df[[f'{c}_hard' for c in TARGET_COLS]].mean() * 100).values

comp_df = pd.DataFrame({
    'Pathology': TARGET_COLS,
    'Gold Standard Prevalence (%)': gold_prev.values.round(1),
    'Full Cohort Pseudo Prevalence (%)': full_prev.round(1)
})

display(comp_df)